In [5]:
import os
import json
import sqlite3
import subprocess
import time
import numpy as np
import pandas as pd

# Paths
aconsole_path = r"C:\Program Files\Aimsun\Aimsun Next 23\aconsole.exe"
base_dir = r"C:\Users\xh8\ORNL_Work\github_workspace\Aimsun-integration-to-RealTwin\Aimsun automation code"
model_dir = os.path.join(base_dir, "Chatt_test", "Model")
model_path = os.path.join(model_dir, "Chatt_test.ang")
script_dir = r"C:\Users\xh8\ORNL_Work\github_workspace\Real-Twin\realtwin\rt_aimsun"
export_script_path = os.path.join(script_dir, "Step6.1_CalibrationInfoExport.py")
assign_script_path = os.path.join(script_dir, "Step6.2_DemandAssign.py")
sqlite_path = os.path.join(model_dir, "Resources", "Outputs", "Chatt_test.sqlite")
info_path = os.path.join(model_dir, "calibration_info.json")
turns_csv = os.path.join(model_dir, "calib_turns.csv")
inflows_csv = os.path.join(model_dir, "calib_inflows.csv")

# Calibration parameters
INFLOW_MAX_VPH = 200          # upper bound of a calibrated entrance flow (veh/h)
population_size = 30          # must be even
num_generations = 10
crossover_rate = 0.75
mutation_rate = 0.1
CalibrationTarget = {"GEH": 5, "GEHPercent": 0.85}


def run_aconsole(cmd):
    print(f"Running command: {' '.join(cmd)}")
    process = subprocess.Popen(cmd,
                               stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT,
                               text=True,
                               encoding="utf-8",
                               errors="replace")
    out, _ = process.communicate()
    return process.returncode, out

In [6]:
t0 = time.time()
import json
input_config = {"AIMSUN": {
    "site_packages": r"C:\Users\xh8\AppData\Local\Programs\Python\Python310\Lib\site-packages"}}

rc, out = run_aconsole([aconsole_path,
                        "-script",
                        export_script_path,
                        model_path])

print(out)
if not (os.path.exists(info_path) and os.path.getmtime(info_path) >= t0 - 1):
    raise RuntimeError(f"calibration info export failed (return code {rc}) - "
                       "calibration_info.json was not written")
if rc != 0:
    print(f"(aconsole exit code {rc} ignored - the export completed)")

with open(info_path) as fh:
    info = json.load(fh)

replication_id = info["replication_id"]
field_df = pd.DataFrame(info["field_approaches"]).rename(columns={"count": "realcount"})
inflow_sections = info["calib_inflow_sections"]

# Group the calibration turnings by approach: one variable per turning,
# normalized within its approach so the percentages sum to exactly 1
approach_groups = []
for turn in info["calib_turns"]:
    key = (turn["junction"], turn["from"])
    if not approach_groups or approach_groups[-1]["key"] != key:
        approach_groups.append({"key": key, "junction": turn["junction"],
                                "from": turn["from"], "tos": []})
    approach_groups[-1]["tos"].append(turn["to"])

n_turn_vars = sum(len(g["tos"]) for g in approach_groups)
n_inflow_vars = len(inflow_sections)
num_variables = n_turn_vars + n_inflow_vars

print("replication id:", replication_id)
print("field approaches:", field_df["section"].tolist())
print(f"{len(approach_groups)} approach group(s) -> {n_turn_vars} turning variable(s)")
print(f"{n_inflow_vars} inflow variable(s): {inflow_sections}")
print(f"total variables: {num_variables}")

Running command: C:\Program Files\Aimsun\Aimsun Next 23\aconsole.exe -script C:\Users\xh8\ORNL_Work\github_workspace\Real-Twin\realtwin\rt_aimsun\Step6.1_CalibrationInfoExport.py C:\Users\xh8\ORNL_Work\github_workspace\Aimsun-integration-to-RealTwin\Aimsun automation code\Chatt_test\Model\Chatt_test.ang
[2026-09-09 15:27:26.464] [info] Command: [C:\Program Files\Aimsun\Aimsun Next 23\aconsole.exe][-script][C:\Users\xh8\ORNL_Work\github_workspace\Real-Twin\realtwin\rt_aimsun\Step6.1_CalibrationInfoExport.py][C:\Users\xh8\ORNL_Work\github_workspace\Aimsun-integration-to-RealTwin\Aimsun automation code\Chatt_test\Model\Chatt_test.ang]
[2026-09-09 15:27:26.575] [info] Platform License
[2026-09-09 15:27:26.584] [info] License Keys to consider: 548880995
[2026-09-09 15:27:26.586] [info] Aimsun Console instance id {f7120ed6-cf66-4c6d-8bae-12bf4058bb7b}
[2026-09-09 15:27:26.586] [info] Aimsun Console PID 28312
[2026-09-09 15:27:26.586] [info] Qt Home C:/Program Files/Aimsun/Aimsun Next 23
[202

In [ ]:
def assignNewTurn(solution):
    """Decode a GA solution into turning-percentage and inflow tables."""
    turn_rows = []
    i = 0
    for group in approach_groups:
        k = len(group["tos"])
        vals = np.asarray(solution[i:i + k], dtype=float)
        i += k
        total = vals.sum()
        if total <= 0:
            vals = np.ones(k)
            total = float(k)
        pcts = vals / total * 100.0
        pcts[-1] = 100.0 - pcts[:-1].sum()    # force the sum to exactly 100
        for to_id, pct in zip(group["tos"], pcts):
            turn_rows.append((group["from"], to_id, pct))
    TurnDf = pd.DataFrame(turn_rows, columns=["entrance", "exit", "turn"])

    inflow_vals = np.asarray(solution[n_turn_vars:], dtype=float) * INFLOW_MAX_VPH
    InflowDf = pd.DataFrame({"entrance": inflow_sections,
                             "flow": np.round(inflow_vals).astype(int)})
    return TurnDf, InflowDf


def genDemand(solution):
    """Write the decoded solution and import it into the traffic states.

    aconsole may crash on exit after finishing (0xC0000374), so success is
    judged by the assign script's ':Assigned' confirmation line, not the
    return code.
    """
    TurnDf, InflowDf = assignNewTurn(solution)
    TurnDf.to_csv(turns_csv, index=False, header=False)
    InflowDf.to_csv(inflows_csv, index=False, header=False)
    rc, out = run_aconsole([aconsole_path, "-script", assign_script_path, model_path])
    if ":Assigned" not in out:
        print(out)
        raise RuntimeError("demand assignment failed (return code %s)" % rc)


def runAimsun():
    """Execute the replication and wait for its SQLite results to be committed."""
    if os.path.exists(sqlite_path):
        try:
            con = sqlite3.connect(sqlite_path)
            for table in ("MISUBPATH", "MISECT"):
                con.execute(f"DELETE FROM {table} WHERE did = ?", (replication_id,))
            con.commit()
            con.close()
        except sqlite3.Error:
            pass

    rc, out = run_aconsole([aconsole_path,
                            "--project",
                            model_path,
                            "--command",
                            "execute",
                            "--target",
                            str(replication_id)])

    deadline = time.time() + 30
    n_rows = 0
    while time.time() < deadline:
        if os.path.exists(sqlite_path):
            try:
                con = sqlite3.connect(sqlite_path)
                n_rows = con.execute("SELECT COUNT(*) FROM MISECT WHERE did = ?",
                                     (replication_id,)).fetchone()[0]
                con.close()
            except sqlite3.Error:
                n_rows = 0
        if n_rows > 0:
            break
        time.sleep(0.5)

    if n_rows == 0:
        print(out)
        raise RuntimeError(f"simulation produced no results after waiting (return code {rc})")


def resultAnalysis():
    """Mean approach-level GEH over the whole simulation period."""
    con = sqlite3.connect(sqlite_path)
    section0 = pd.read_sql_query(
        "SELECT oid, count FROM MISECT WHERE did = %d AND sid = 0 AND ent = 0"
        % replication_id, con)
    con.close()
    section = section0.drop_duplicates(subset="oid", keep="last")
    compare = field_df.merge(section, left_on="section", right_on="oid", how="left")
    compare = compare.dropna(subset=["count"])
    compare["GEH"] = np.sqrt(2 * ((compare["count"] - compare["realcount"]) ** 2)
                             / (compare["count"] + compare["realcount"]))
    meanGEH = compare["GEH"].mean()
    GEHPercent = (compare["GEH"] < CalibrationTarget["GEH"]).mean()
    return meanGEH, GEHPercent

In [ ]:
# Optional smoke test: one evaluation with a mid-range solution
test_solution = np.full(num_variables, 0.5)
genDemand(test_solution)
runAimsun()
meanGEH, GEHPercent = resultAnalysis()
print("mean GEH = %.3f   (%.0f%% of approaches < %d)"
      % (meanGEH, GEHPercent * 100, CalibrationTarget["GEH"]))

In [ ]:
from mealpy import FloatVar, GA

best_txt_path = os.path.join(base_dir, "GA_GEH_best.txt")
eval_count = [0]
best_so_far = [np.inf]

def objective_function(x):
    genDemand(x)
    runAimsun()
    value, _ = resultAnalysis()
    eval_count[0] += 1
    # Report and save only when a better solution is found
    if value < best_so_far[0]:
        best_so_far[0] = value
        print("eval %d: new best mean GEH = %.3f" % (eval_count[0], value))
        np.savetxt(best_txt_path, np.asarray(x, dtype=float), fmt="%f",
                   header="mean GEH = %.6f (evaluation %d)" % (value, eval_count[0]))
    return value


problem_dict = {
    "obj_func": objective_function,
    "bounds": FloatVar(lb=[0.0] * num_variables, ub=[1.0] * num_variables),
    "minmax": "min",
    "verbose": True,
    "save_population": True,
}

optimizer = GA.BaseGA(
    epoch=num_generations,
    pop_size=population_size,
    pc=crossover_rate,
    pm=mutation_rate,
)

start_time = time.time()
g_best = optimizer.solve(problem_dict, mode="single")
print("elapsed: %.1f min" % ((time.time() - start_time) / 60))

best_solution = g_best.solution
best_fitness = g_best.target.fitness
print("Best solution:", best_solution)
print("Best mean GEH:", best_fitness)
np.savetxt(best_txt_path, np.asarray(best_solution, dtype=float), fmt="%f",
           header="mean GEH = %.6f (final)" % best_fitness)

In [ ]:
# Final run with the best solution: the calibrated demand stays in the .ang,
# and calib_turns.csv / calib_inflows.csv hold the decoded values.
# If the GA was interrupted, the best solution found so far is recovered
# from GA_GEH_best.txt (it is updated whenever a better one is found).
try:
    best_solution
except NameError:
    best_solution = np.loadtxt(os.path.join(base_dir, "GA_GEH_best.txt"))
    print("loaded best solution from GA_GEH_best.txt")

genDemand(best_solution)
runAimsun()
meanGEH, GEHPercent = resultAnalysis()
print("Final mean GEH = %.3f" % meanGEH)
print("In final results, %.2f percent of approaches have GEH lower than %d."
      % (GEHPercent * 100, CalibrationTarget["GEH"]))

TurnDf, InflowDf = assignNewTurn(best_solution)
print("\nCalibrated inflows (veh/h):")
print(InflowDf.to_string(index=False))
print("\nCalibrated turning percentages:")
print(TurnDf.to_string(index=False))